# Pizza Shop Voice Agent

In this notebook we deploy and test the pizza shop voice agent application.

The app connects the speech models from the previous notebook to an LLM agent that can:

- **Take pizza orders** — choose pizza type, toppings, quantities
- **Calculate order totals** — pricing and order summary
- **Handle delivery** — delivery address and timing

The architecture follows the **voice sandwich** pattern:

```
Browser Mic -> [Whisper STT] -> Text -> [LLM Agent Graph] -> Text -> [Higgs-Audio TTS] -> Speaker
```

## Prerequisites

Before running this notebook, make sure:

1. You have completed the **Models** notebook (speech models are deployed and running)
2. You are logged into OpenShift — run the cell below (replace `<username>` and `<password>`):

In [ ]:
!oc login --server=https://api.ocp.cloud.rhai-tmm.dev:6443 -u <username> -p <password>
!oc project ai-roadshow

Verify you are logged in:

In [ ]:
!oc whoami
!oc project

## Verify Speech Models

Check that both speech models from the previous notebook are running:

In [ ]:
!echo '=== Whisper (STT) ==='
!oc get pods -l app.kubernetes.io/name=whisper
!echo ''
!echo '=== Higgs-Audio (TTS) ==='
!oc get pods -l app=higgs-audio-predictor

## Deploy the Voice Agent App

The app is packaged as a Helm chart with two components:

| Component | Description | Port |
|-----------|-------------|------|
| **Backend** | Python WebSocket server (LangGraph agent) | 8765 |
| **Frontend** | Next.js UI served by Nginx | 8080 |

The backend needs credentials to access the LLM (Llama 4 Scout on MaaS) and the speech models.

### Gather credentials

First, collect the endpoint URLs and tokens the backend needs.

The MaaS gateway uses Kuadrant/Authorino auth - you need a MaaS token to proceed.

In [ ]:
import subprocess, base64

# LLM — Llama 4 Scout via MaaS
llm_model = "llama-4-scout-17b-16e-w4a16"
llm_base_url = f"https://maas.apps.ocp.cloud.rhai-tmm.dev/prelude-maas/{llm_model}/v1"

# MaaS token
llm_token = "<your maas token>"

if not llm_token or "error" in llm_token.lower():
    print(f"ERROR: Could not create LLM token. Check your SA name: {maas_sa}")
    print("Run: oc get sa -n maas-default-gateway-tier-enterprise")
else:
    print(f"LLM Token:     {llm_token[:20]}...")

# STT — Whisper
whisper_urls = subprocess.run(
    ["oc", "get", "llmisvc", "whisper", "-o",
     "jsonpath={.status.addresses[?(@.name=='gateway-external')].url}"],
    capture_output=True, text=True
).stdout.strip().split(" ")
whisper_url = [u for u in whisper_urls if u.startswith("https")][-1]
stt_url = f"{whisper_url}/v1/audio/transcriptions"

stt_token_b64 = subprocess.run(
    ["oc", "get", "secret", "whisper-sa-whisper-sa", "-o",
     "jsonpath={.data.token}"],
    capture_output=True, text=True
).stdout.strip()
stt_token = base64.b64decode(stt_token_b64).decode()

# TTS — Higgs-Audio (cluster-internal)
tts_url = "http://higgs-audio-predictor:8080/v1"
tts_model = "higgs-audio-v2-generation-3B-base"

print(f"LLM Base URL:  {llm_base_url}")
print(f"STT URL:       {stt_url}")
print(f"STT Token:     {stt_token[:20]}...")
print(f"TTS URL:       {tts_url}")
print(f"TTS Model:     {tts_model}")

### Helm install

Deploy the app with the collected credentials. This creates the backend and frontend Deployments,
Services, Secret, and an OpenShift Route.

In [ ]:
import subprocess, shlex

helm_cmd = [
    "helm", "upgrade", "--install", "ai-voice-agent",
    "../../ai-voice-agent/deploy/chart",
    "--namespace", "ai-roadshow",
    "--set", f"backend.env.MODEL_NAME={llm_model}",
    "--set", f"backend.env.BASE_URL={llm_base_url}",
    "--set", f"backend.env.TTS_URL={tts_url}",
    "--set", f"backend.env.TTS_MODEL={tts_model}",
    "--set", f"backend.env.STT_URL={stt_url}",
    "--set", "backend.env.STT_MODEL=whisper",
    "--set", f"backend.secret.API_KEY={llm_token}",
    "--set", f"backend.secret.STT_TOKEN={stt_token}",
    "--set", "mlflow.enabled=false",
    "--set", "guardrails.enabled=false",
    "--set", "nemoGuardrails.enabled=false",
]

result = subprocess.run(helm_cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

### Wait for pods

Wait until both backend and frontend pods are running. The backend pulls a container image
and starts the WebSocket server — this can take a minute or two on first deploy.

In [ ]:
!echo '=== Backend ==='
!oc rollout status deployment/ai-voice-agent-backend --timeout=120s 2>&1
!echo ''
!echo '=== Frontend ==='
!oc rollout status deployment/ai-voice-agent-frontend --timeout=120s 2>&1
!echo ''
!oc get pods -l 'app.kubernetes.io/part-of=ai-voice-agent'

### Get the app URL

The Helm chart creates an OpenShift Route that exposes the frontend with TLS.
Nginx proxies WebSocket connections from `/ws` to the backend.

In [ ]:
import subprocess

route_host = subprocess.run(
    ["oc", "get", "route", "ai-voice-agent", "-o", "jsonpath={.spec.host}"],
    capture_output=True, text=True
).stdout.strip()

app_url = f"https://{route_host}"
ws_url = f"wss://{route_host}/ws"

print(f"App URL:       {app_url}")
print(f"WebSocket URL: {ws_url}")

## Test the App

### Quick test via WebSocket

Before opening the UI, let's verify the backend is working by sending a text message
directly over the WebSocket. This tests the full pipeline:
text input -> LLM agent -> TTS audio output.

In [ ]:
import asyncio, json, ssl

try:
    import websockets
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "websockets"], check=True)
    import websockets

ssl_ctx = ssl.create_default_context()
ssl_ctx.check_hostname = False
ssl_ctx.verify_mode = ssl.CERT_NONE

async def test_text_message():
    async with websockets.connect(ws_url, ssl=ssl_ctx) as ws:
        # Send a text message (no mic needed)
        msg = {"type": "text", "text": "Hi, I'd like to order a pizza please"}
        await ws.send(json.dumps(msg))
        print(f"Sent: {msg['text']}")
        print("---")

        audio_chunks = 0
        while True:
            try:
                response = await asyncio.wait_for(ws.recv(), timeout=60)
                if isinstance(response, bytes):
                    audio_chunks += 1
                    continue
                data = json.loads(response)
                msg_type = data.get("type", "")
                if msg_type == "graph_result":
                    for m in data.get("messages", []):
                        role = m.get("role", "")
                        content = m.get("content", "")
                        if role == "human":
                            continue
                        print(f"  [{role}] {content}")
                elif msg_type == "tts_begin":
                    print(f"  TTS streaming started...")
                elif msg_type == "tts_end":
                    audio_seconds = (audio_chunks * 960) / (24000 * 2)
                    print(f"  TTS complete ({audio_chunks} chunks, ~{audio_seconds:.1f}s audio)")
                    break
                elif msg_type == "error":
                    print(f"  ERROR: {data.get('error', '')}")
                    break
            except asyncio.TimeoutError:
                print("  (timeout waiting for response)")
                break

await test_text_message()

### Open the Web UI

Open the app URL in your browser to try the full voice experience:

1. Click the link below to open the pizza shop UI
2. The UI auto-connects to the WebSocket backend
3. Click **Start** to begin recording with your microphone
4. Speak naturally — "I'd like to order a large pepperoni pizza"
5. The agent will respond with spoken audio

You can also use the text input box for quick testing without a microphone.

In [ ]:
from IPython.display import display, Markdown

display(Markdown(f"**Open the Pizza Shop UI:** [{app_url}]({app_url})"))

### Check backend logs

If something isn't working, check the backend logs for errors:

In [ ]:
!oc logs deployment/ai-voice-agent-backend --tail=30

## Cleanup (optional)

To remove the pizza shop app (keeps the speech models):

In [ ]:
# Uncomment to remove the app:
# !helm uninstall ai-voice-agent -n ai-roadshow

## Summary

You have deployed and tested the pizza shop voice agent:

| Component | What it does |
|-----------|-------------|
| **Backend** | Python WebSocket server running a LangGraph agent with 4 specialist agents (supervisor, pizza, order, delivery) |
| **Frontend** | Next.js UI with microphone capture, real-time audio streaming, and a talking pizza animation |
| **Nginx** | Proxies WebSocket connections from the browser to the backend |

The agent graph routes conversations through specialist agents:

```
User -> Supervisor -> Pizza Agent    (choose pizza type & toppings)
                   -> Order Agent    (calculate totals)
                   -> Delivery Agent (handle delivery details)
```

Next steps:
- Explore the agent prompts and graph in `ai-voice-agent/backend/src/`
- Add observability with MLflow (re-deploy with `mlflow.enabled=true`)
- Enable guardrails for content safety